In [1]:
# path setup: this notebook now lives in revision_scripts/, while the data paths (data/...) and the
# config / netmedgpt imports are relative to the repository root. Safe to re-run; delete if the
# notebook is moved back to the root.
import os, sys
if os.path.basename(os.getcwd()) == 'revision_scripts':
    os.chdir('..')
for p in (os.getcwd(), os.path.join(os.getcwd(), 'revision_scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)


# Walk-embedding baselines on the never-walked edges: training code

Training code of the walk-embedding baselines for check 3+ (`node2vec_classifier`, `node2vec_rel_classifier`,
`dreamwalk_samecorpus`, `dreamwalk_full`), trained on the same walks as NetMedGPT with the never-walked edges removed
from the classifiers' training pairs. This notebook only defines the settings and functions; they are used by
`check3_paper_metrics_baselines.ipynb`, which evaluates the baselines with the benchmark protocol.

Requires the `walk_baselines` kernel (gensim, xgboost).

In [2]:
INFERENCE = "random_link_split"
SEEDS = [2, 3, 4, 5]
BASELINES = ['node2vec_classifier', 'node2vec_rel_classifier', 'dreamwalk_samecorpus', 'dreamwalk_full']

OUT_DIR = "data/new_analysis/coverage"
# saved DREAMwalk embeddings (dreamwalk_samecorpus.py / dreamwalk_full.py)
DREAMWALK_EMB = {"dreamwalk_samecorpus": "data/new_analysis/dreamwalk_samecorpus/{inference}_seed{seed}/embedding_file.pkl",
                 "dreamwalk_full": "data/new_analysis/dreamwalk_full/{inference}_seed{seed}/embedding_file.pkl"}
# skip-gram settings of node2vec_baseline.py (defaults used for the walk-baseline results)
N2V_PARAMS = dict(vector_size=300, window=4, sg=1, negative=5, min_count=1, epochs=5, workers=30)
PAIR_BUDGET = 500000         # (head, candidate) pairs per scoring call

DEGREE_BINS = [1, 2, 6, 21, 101, 1001, float("inf")]
DEGREE_LABELS = ["1", "2-5", "6-20", "21-100", "101-1000", ">1000"]
CHUNK = 1000000

In [3]:
# graph lookups, walk statistics, coverage and check 3+ ranking of walk_coverage_analysis.ipynb (function cells only)
import pickle
import nbformat
from common import walks_to_sentences, embedding_matrix

FUNCTION_CELLS = ("import os", "# Key helpers", "# Training graph", "# Walk statistics", "import scipy.sparse",
                  "from netmedgpt.model")
loaded = 0
for cell in nbformat.read("revision_scripts/walk_coverage_analysis.ipynb", as_version=4).cells:
    if cell.cell_type == "code" and cell.source.startswith(FUNCTION_CELLS):
        exec(cell.source)
        loaded += 1
assert loaded == len(FUNCTION_CELLS), f"found {loaded} of {len(FUNCTION_CELLS)} function cells"


# score matrix from a baseline pair_score_fn(heads, tails, relation_token) (common.py / baseline_utils.py interface)
def pair_score_matrix_fn(pair_score_fn, z, cand):
    cand_cpu = cand.cpu()

    def fn(hb):
        heads = torch.as_tensor(hb, dtype=torch.long).repeat_interleave(len(cand_cpu))
        tails = cand_cpu.repeat(len(hb))
        s = pair_score_fn(heads, tails, z)
        return torch.as_tensor(s).float().cpu().numpy().reshape(len(hb), len(cand_cpu))
    return fn


# walk baselines trained on the same corpus (node2vec_baseline.py / baseline_utils.py code); the never-walked edges
# (exclude: relation -> pair keys) are removed from the classifier training pairs
def build_baselines(split, g, data, exclude, names, device, seed):
    train = split['train_txgnn']
    keys = pair_key(train['x_index'].to_numpy(), train['y_index'].to_numpy(), g['n_nodes'])
    rel = train['relation'].to_numpy()
    drop = np.zeros(len(train), dtype=bool)
    for relation, ex in exclude.items():
        drop |= (rel == relation) & np.isin(keys, ex)
    split_b = dict(split, train_txgnn=train[~drop])
    print(f"Baselines: {int(drop.sum())} never-walked edges removed from the classifier training pairs")

    scorers = {}
    n2v = [(False, 'node2vec_classifier'), (True, 'node2vec_rel_classifier')]
    if any(name in names for _, name in n2v):
        try:
            from gensim.models import Word2Vec
            from node2vec_baseline import fit_logreg_per_relation
        except ImportError as err:
            print(f"node2vec baselines skipped ({err}); use the walk_baselines environment")
        else:
            for keep_relations, name in n2v:
                if name not in names:
                    continue
                print(f"Training {name}...")
                sentences = walks_to_sentences(data['node2vec_walks'], keep_relations=keep_relations)
                w2v = Word2Vec(sentences, seed=seed, **N2V_PARAMS)
                E, _ = embedding_matrix(w2v.wv, split['vocab_size'], N2V_PARAMS['vector_size'])
                scorers[name] = fit_logreg_per_relation(split_b, E, device, seed)
                del sentences, w2v

    for name, emb_path in DREAMWALK_EMB.items():
        if name not in names:
            continue
        path = emb_path.format(inference=split['inference'], seed=seed)
        try:
            from baseline_utils import fit_xgb_per_relation
            import xgboost  # noqa: F401
        except ImportError as err:
            print(f"{name} skipped ({err}); use the walk_baselines environment")
            continue
        if not os.path.exists(path):
            print(f"{name} skipped: {path} not found")
            continue
        print(f"Training {name} XGBoost classifiers...")
        with open(path, "rb") as f:
            embeddings = pickle.load(f)
        E, _ = embedding_matrix(embeddings, split['vocab_size'], len(next(iter(embeddings.values()))))
        scorers[name] = fit_xgb_per_relation(split_b, E, seed, n_jobs=N2V_PARAMS['workers'])
    return scorers


# never-walked training edges of every queried relation (query direction), as pair keys
def never_walked(split, g, st):
    cov_und = covered_undirected(g, st)
    exclude = {}
    for r, relation in enumerate(split['queried_edge_types']):
        head_type, tail_type = split['queried_node_types'][r].split('|')
        pos = training_positives_by_coverage(split, g, cov_und, relation)
        pos = pos[(g['node_type'][pos['x_index'].to_numpy()] == head_type)
                  & (g['node_type'][pos['y_index'].to_numpy()] == tail_type) & ~pos['seen']]
        exclude[relation] = pair_key(pos['x_index'].to_numpy(), pos['y_index'].to_numpy(), g['n_nodes'])
    return exclude


# scores the edges of check 3+ (NetMedGPT rows of check3x_edges.csv) with every baseline
def score_baselines(split, data, nm_edges, names, device):
    g = build_graph(split)
    st = WalkStats(g)
    walks = data['node2vec_walks'].numpy()
    for i in range(0, len(walks), CHUNK):
        st.add(walks[i:i + CHUNK])
    scorers = build_baselines(split, g, data, never_walked(split, g, st), names, device, split['seed'])

    out = []
    edge = split['edge']
    for r, relation in enumerate(split['queried_edge_types']):
        e = nm_edges[nm_edges['relation'] == relation][['group', 'head', 'tail', 'head_degree', 'tail_degree']]
        e = e.reset_index(drop=True)
        z = int(edge.loc[edge['relation'] == relation, 'z_index'].unique().item())
        cand = split['all_ids_node2'][z]
        known = kg_tails(split, relation, split['queried_node_types'][r])
        for method, pair_fn in scorers.items():
            fn = pair_score_matrix_fn(pair_fn, z, cand)
            auc, hit = edge_ranks(fn, e, cand, known, max(1, PAIR_BUDGET // len(cand)))
            res = e.assign(relation=relation, method=method, auc_edge=auc, hit100=hit)
            out.append(res)
            print(f"{relation} | {method}\n{res.groupby('group', sort=False)[['auc_edge', 'hit100']].mean().round(4).to_string()}")
    return pd.concat(out, ignore_index=True)